In [ ]:
# agent_faithful_optimized.py
import os
import time
import numpy as np
import warnings

from pydantic import BaseModel
from langgraph.graph import StateGraph
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import Tool
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# avoid streamlit/pytorch compatibility problems (kept from your original)
import torch

warnings.filterwarnings("ignore")
torch.classes.__path__ = []
os.environ["TOKENIZERS_PARALLELISM"] = "false"

c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
llm = ChatOpenAI(
    model_name="hermes-3-llama-3.2-3b@q6_k",
    openai_api_base="http://127.0.0.1:1234/v1",
    openai_api_key="lm-studio",
    temperature=0.3,
    max_tokens=256
)

# exploratory LLM used to generate diverse candidate answers
llm_exploratory = ChatOpenAI(
    model_name="hermes-3-llama-3.2-3b@q6_k",
    openai_api_base="http://127.0.0.1:1234/v1",
    openai_api_key="lm-studio",
    temperature=0.9,
    max_tokens=256
)

embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en")

vector_db = Chroma(
    persist_directory="rag/chroma_db",
    embedding_function=embedding_model
)
retriever = vector_db.as_retriever()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5839.03it/s]


In [3]:
# Prompt / combine chain
prompt = PromptTemplate.from_template(
    """
        You are a specialist assistant in public procurement.
        Answer the following question, in Brazilian Portuguese, based on the documents provided:

        {context}

        Question: {input}
        Remember that for any question that deviates from this topic, you must state that you cannot help with other topics because you specialize in procurement and not other matters.
    """
)

combine_docs_chain = (
    prompt
    | llm
    | StrOutputParser()
)

# qa_chain expects a concatenated context string
qa_chain = (
    {
        "context": RunnablePassthrough(),
        "input": RunnablePassthrough()
    }
    | (prompt | llm | StrOutputParser())
)

In [4]:
# Web search tool (no create_agent)
search = DuckDuckGoSearchAPIWrapper(region="br-pt", max_results=5)
web_search_tool = Tool(
    name="WebSearch",
    func=search.run,
    description="Search updated information on internet about public digging"
)

# Note: we DO NOT use create_agent; we'll call search.run() directly in the node.

In [5]:
class AgentState(BaseModel):
    query: str
    next_step: str = ""
    retrieved_info: list = []
    possible_responses: list = []
    similarity_scores: list = []
    ranked_response: str = ""
    confidence_score: float = 0.0

In [6]:
def agent_decision_step(state: AgentState) -> AgentState:
    q = state.query.lower()
    if any(p in q for p in ["explain", "summarize", "define", "concept", "general", "what is"]):
        state.next_step = "generate"
    elif any(p in q for p in ["search on web", "news", "updated", "recently", "latest informations"]):
        state.next_step = "use_web"
    else:
        state.next_step = "retrieve"
    return state

In [7]:
def use_web_tool(state: AgentState) -> AgentState:
    """
        Directly calls the DuckDuckGo tool and formats the output.
        Adjust if your DuckDuckGo wrapper version uses a different method (e.g., .run vs. .invoke).
    """
    try:
        result = search.run(state.query) 
    except Exception as e:
        result = f"Error to execute the search: {e}"
    # normalize result to string
    if isinstance(result, dict):
        # try common keys
        out = result.get("text") or result.get("output") or str(result)
    else:
        out = str(result)
    state.ranked_response = out or "No information was obtained through web search."
    state.confidence_score = 0.5
    return state